In [1]:
import torch
from torch import nn
from torch.utils import data
batch_size=10
lr=3e-2
epoch=10
device=torch.device('cuda')

In [2]:
def corr2d(x,k):
    h,w=k.size()
    y=torch.zeros(size=(x.size(0)-h+1,x.size(1)-w+1))
    for i in range(y.size(0)):
        for j in range(y.size(1)):
            y[i,j]=(x[i:i+h,j:j+w]*k).sum()
    return y

In [3]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]])

In [4]:
class conv2d(nn.Module):
    def __init__(self,mysize):
        super().__init__()
        self.weight=nn.Parameter(torch.rand(size=mysize))
        self.bias=nn.Parameter(torch.zeros(1))
    def forward(self,x):
        return corr2d(x,self.weight)+self.bias

In [5]:
X = torch.ones((6, 8))
X[:, 2:6] = 0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

In [6]:
k=torch.tensor([1.0,-1.0]).reshape(1,2)
corr2d(X,k)

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

In [7]:
def Corr2d(x,k):
    h,w=k.size()
    y=torch.zeros(size=(x.size(0),x.size(1)-h+1,x.size(2)-w+1))
    for i in range(y.size(1)):
        for j in range(y.size(2)):
            y[:,i,j]=(x[:,i:i+h,j:j+w]*k).sum(dim=(1,2))
    return y

In [8]:
def data_maker():
    X = torch.randn((100,6, 8))
   # X[:, :,2:6] = 0
    true_k=torch.tensor([[1.0,-1.0]])
    Y=Corr2d(X,true_k)
    X=X.reshape((100,1,6,8))
    Y=Y.reshape((100,1,6,7))
    return X,Y

In [9]:
def data_loader(data_arry,batch_size,shuffle):
    dataset=data.TensorDataset(*(data_arry))
    return data.DataLoader(dataset,batch_size,shuffle=shuffle)


In [10]:
def init(m):
    if isinstance(m,nn.Conv2d):
        nn.init.xavier_uniform_(m.weight)

In [11]:
features,labels=data_maker()
train_iter=data_loader((features,labels),batch_size,True)

net=nn.Conv2d(in_channels=1,out_channels=1,kernel_size=(1,2),bias=False).to(device)
net.apply(init)
loss=nn.MSELoss()
trainer=torch.optim.SGD(net.parameters(),lr=lr)

for i in range(epoch):
    net.train()
    train_loss=0.0
    train_count=0
    for x,y in train_iter:
        x=x.to(device)
        y=y.to(device)
        trainer.zero_grad()
        l=loss(net(x),y)
        train_loss+=l.item()*y.size(0)
        train_count+=y.size(0)
        l.backward()
        trainer.step()
    print(f"训练轮次：{i+1},损失：{train_loss/train_count:.5f}")

net.eval()
print(net.weight.detach())
print(net.weight.size())


训练轮次：1,损失：1.08122
训练轮次：2,损失：0.31479
训练轮次：3,损失：0.09170
训练轮次：4,损失：0.02669
训练轮次：5,损失：0.00778
训练轮次：6,损失：0.00227
训练轮次：7,损失：0.00066
训练轮次：8,损失：0.00019
训练轮次：9,损失：0.00006
训练轮次：10,损失：0.00002
tensor([[[[ 0.9972, -0.9999]]]], device='cuda:0')
torch.Size([1, 1, 1, 2])


In [13]:
torch.save(net.state_dict(),'conv2d-data')
net1=nn.Conv2d(in_channels=1,out_channels=1,kernel_size=(1,2),bias=False)
state=torch.load('conv2d-data')
net1.load_state_dict(state)
x = torch.ones((6, 8)).reshape(1,1,6,8)
x[:,:,:,2:6] = 0
print(net1(x))

tensor([[[[-0.0027,  0.9972,  0.0000,  0.0000,  0.0000, -0.9999, -0.0027],
          [-0.0027,  0.9972,  0.0000,  0.0000,  0.0000, -0.9999, -0.0027],
          [-0.0027,  0.9972,  0.0000,  0.0000,  0.0000, -0.9999, -0.0027],
          [-0.0027,  0.9972,  0.0000,  0.0000,  0.0000, -0.9999, -0.0027],
          [-0.0027,  0.9972,  0.0000,  0.0000,  0.0000, -0.9999, -0.0027],
          [-0.0027,  0.9972,  0.0000,  0.0000,  0.0000, -0.9999, -0.0027]]]],
       grad_fn=<ConvolutionBackward0>)
